### Prediction

In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

#### Load the trained model, scalar pickle, onehot encoder

In [21]:
## Load trained model
model = load_model('../artifacts/model.h5')

## Load the encoder and scalar
with open('../artifacts/gender_label_encoder.pkl', 'rb') as file:
    gender_le = pickle.load(file)

with open('../artifacts/geo_onehot_encoder.pkl', 'rb') as file:
    geo_ohe = pickle.load(file)

with open('../artifacts/scalar.pkl', 'rb') as file:
    scaler = pickle.load(file)

#### Input data for prediction

In [22]:
## Example of input data
input_data = {
    'CreditScore': 600,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 40,
    'Tenure': 3,
    'Balance': 60000,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000
}


In [23]:
## Create dataframe
input_data_df = pd.DataFrame(data=[input_data])
input_data_df

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,600,France,Male,40,3,60000,2,1,1,50000


In [24]:
## Apply encoding on Geography cols
geo_encoded = geo_ohe.transform(input_data_df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=geo_ohe.get_feature_names_out())
geo_encoded_df

,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [25]:
## Merge encoded cols to original data
input_data_df = pd.concat([input_data_df.drop(['Geography'], axis=1), geo_encoded_df], axis=1)
input_data_df

,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,Male,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [26]:
## Apply encoding on Gender col
input_data_df['Gender'] = gender_le.transform(input_data_df[['Gender']])
input_data_df

e:\AI\03 - Deep Learning\Deep-Learning-Projects\ANN\Churn-Modelling\churn_venv\Lib\site-packages\sklearn\preprocessing\_label.py:139: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, dtype=self.classes_.dtype, warn=True)


,CreditScore,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,600,1,40,3,60000,2,1,1,50000,1.0,0.0,0.0


In [27]:
## Apply Scaling
input_data_df = scaler.transform(input_data_df)
input_data_df

array([[-0.53598516,  0.91324755,  0.10479359, -0.69539349, -0.25781119,
         0.80843615,  0.64920267,  0.97481699, -0.87683221,  1.00150113,
        -0.57946723, -0.57638802]])

#### Prediction

In [29]:
## Get the prediction probability
prediction_prob = model.predict(input_data_df)
prediction_prob

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step


array([[0.03101601]], dtype=float32)

In [30]:
possible_results = ['The customer is not likely to churn', 'The customer is likely to churn']
prediction_result = [possible_results[0] if prediction_prob<0.5 else possible_results[1]]
prediction_result

['The customer is not likely to churn']